In [1]:
# numpy(넘파이)는 숫자 여러 개를 '배열(array)'로 묶어 한 번에 계산하게 해 주는 라이브러리
import numpy as np

# torch는 PyTorch 라이브러리
# 자동미분(autograd)과 optimizer(torch.optim.SGD)를 쓰기 위해 사용
# - 자동 미분 : 사람이 미분 공식을 적지 않아도 PyTorch가 a.grad, b.grad를 계산해 주는 기능
# - optimizer : 우리가 직접 하던 a, b 업데이트를 PyTorch가 대신 수행해 주는 도구
import torch
from math import e

In [2]:
# 입력값 X와 정답 y 준비
X = np.array([160, 170, 180, 190])
y = np.array([0, 0, 1, 1])
print("입력값 X:", X)
print("정답값 y:", y)

입력값 X: [160 170 180 190]
정답값 y: [0 0 1 1]


In [3]:
# 입력값 정규화
# 평균(mean)과 표준편차(std)를 계산
X_mean = np.mean(X)
X_std = np.std(X)

# 정규화 공식: (원본값 - 평균) / 표준편차
X_norm = (X - X_mean) / X_std

print("입력값 평균 X_mean:", X_mean)
print("입력값 표준편차 X_std:", X_std)
print("정규화된 입력값 X_norm:", X_norm)

입력값 평균 X_mean: 175.0
입력값 표준편차 X_std: 11.180339887498949
정규화된 입력값 X_norm: [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [4]:
# X_norm과 y를 PyTorch tensor로 변환
# 학습에 사용할 입력값(X_norm)과 정답(y)을 tensor로 바꿔 둡니다
# dtype = torch.float32 : 소수점 계산(미분)을 위해 실수(float)형식으로 만듭니다
X_norm_tensor = torch. tensor(X_norm, dtype=torch.float32)
y_tensor = torch. tensor(y, dtype=torch.float32)

print("학습용 입력 tensor X_norm_tensor:", X_norm_tensor)
print("학습용 정답 tensor y_tensor:", y_tensor)

학습용 입력 tensor X_norm_tensor: tensor([-1.3416, -0.4472,  0.4472,  1.3416])
학습용 정답 tensor y_tensor: tensor([0., 0., 1., 1.])


In [5]:
# 가중치 a와 편향 b 초기값 설정 (requires_grad=True 인 tensor)

# a는 가중치(weight), b는 편향(bias)입니다.
# a는 원래 키(cm)가 아니라 정규화된 입력값 X_norm에 곱해지는 값입니다.
a = torch.tensor(0.1, dtype=torch.float32, requires_grad=True)
b = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

# .item()으로 숫자만 깔끔하게 꺼내 출력
print("초기 가중치 a:", a.item())
print("초기 편향 b:", b.item())

초기 가중치 a: 0.10000000149011612
초기 편향 b: 0.0


In [6]:
# 학습 설정 (learning_rate, epochs)
learning_rate = 0.1
epochs = 1000
print("learning_rate:", learning_rate)
print("epochs:", epochs)

learning_rate: 0.1
epochs: 1000


In [7]:
# optimizer 생성 (이번 실습의 핵심)

# torch.optim.SGD([a, b], lr=learning_rate)
# - [a, b] : optimizer가 업데이트할 학습 대상(파라미터)입니다.
#            optimizer는 a.grad, b.grad를 이용해 a, b를 업데이트 합니다.
# - lr=learning_rate : 한 번에 얼마나 움직일지(학습률). 위에서 정한 0.1을 넘겨줍니다.
#
# 이 optimizer가 학습 루프에서 하는 일은 단 두 가지입니다.
# optimizer.zero_grad() : a.grad, b.grad를 0으로 초기화 (이전의 a.grad.zero_(), b.grad.zero_() 역할)
# optimizer.step() : a, b를 업데이트 (이전의 a -= lr*a.grad, b -= lr*b.grad 역할)
optimizer = torch.optim.SGD([a, b], lr=learning_rate)

print("optimizer 준비 완료:", optimizer)

optimizer 준비 완료: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [8]:
# 시그모이드 함수 정의
def sigmoid(H):
    return 1 / (1 + e**(-H))

In [9]:
# 학습 전 예측 확인
H = a.item() * X_norm + b.item()

z = sigmoid(H)

print("학습 전 선형 계산값 H(x):", H)
print("학습 전 시그모이드 예측 확률 z:", z)

학습 전 선형 계산값 H(x): [-0.13416408 -0.04472136  0.04472136  0.13416408]
학습 전 시그모이드 예측 확률 z: [0.4665092  0.48882152 0.51117848 0.5334908 ]


In [10]:
# 학습 전 비용(Cost) 계산
epsilon = 1e-7
z_safe = np.clip(z, epsilon, 1 - epsilon)

costs = -(y * np.log(z_safe) + (1 - y) * np.log(1 - z_safe))
Cost = np.mean(costs)

print("학습 전 각 샘플의 비용(Cost):", costs)
print("학습 전 평균 비용(Cost):", Cost)

학습 전 각 샘플의 비용(Cost): [0.62831345 0.67103648 0.67103648 0.62831345]
학습 전 평균 비용(Cost): 0.6496749672265923


In [11]:
# optimizer로 경사 하강법 학습 (이번 실습의 핵심 루프)

# 한 번의 epoch에서 일어나는 단계
# 1. optimizer.zero_grad()  : 이전 epoch의 grad를 0으로 초기화
# 2. H = a*X_norm + b       : 선형 계산값
# 3. z = sigmoid(H)         : 예측 확률
# 4. z_safe = clamp(z)      : log(0) 방지를 위해 z 범위 제한
# 5. Cost = BCE(z_safe, y)  : 비용 계산
# 6. Cost.backward()        : a.grad, b.grad 자동 계산
# 7. optimizer.step()       : a, b 업데이트
# 8. 학습 상태 출력
for epoch in range(epochs):
    # 이전에 계산된 grad초기화
    # PyTorch의 grad는 덮어쓰기가 아니라 '누적(더하기)'됩니다
    # 이전의 grad가 남아있으면 이번 grad와 더해져 잘못된 업데이트가 됩니다
    optimizer.zero_grad()

    # 선형 계산
    H = a * X_norm_tensor + b

    # sigmoid를 적용해 예측 확률 z 계산
    z = torch.sigmoid(H)

    # BCE Cost 계산에서 log(0)이 발생하지 않도록 z 범위 제한
    z_safe = torch.clamp(z, epsilon, 1 - epsilon)

    # BCE : Binary Cross Entropy Cost 계산
    costs = -(
        y_tensor * torch.log(z_safe)
        + (1 - y_tensor) * torch.log(1 - z_safe)
    )
    mean_cost = torch.mean(costs)

    # backward: a.grad, b.grad 자동 계산
    mean_cost.backward()

    # optimizer가 a와 b 업데이트
    # with torch.no_grad(): 
    #   a = a - learning_rate * grad_a
    #   b = b - learning_rate * grad_b
    # 를 한줄로 대체
    optimizer.step()

    # 학습상태 출력
    if epoch % 100 == 0 or epoch == epochs -1:
        print(
            f"Epoch {epoch}, "
            f"평균 비용(Cost): {mean_cost.item():.6f}, "
            f"a: {a.item():.6f}, "
            f"b: {b.item():.6f}"
        )
    
    # 초반 3 epoch에서만 a.grad, b.grad 값 확인해보기
    if epoch < 3:
        grad_a = a.grad.item()
        grad_b = b.grad.item()
        print(
            f"  (확인용) a.grad={grad_a:.6f}, "
            f"b.grad={grad_b:.6f}"
        )

Epoch 0, 평균 비용(Cost): 0.649675, a: 0.142225, b: -0.000000
  (확인용) a.grad=-0.422248, b.grad=0.000000
  (확인용) a.grad=-0.411755, b.grad=0.000000
  (확인용) a.grad=-0.401573, b.grad=0.000000
Epoch 100, 평균 비용(Cost): 0.198225, a: 2.069146, b: 0.000000
Epoch 200, 평균 비용(Cost): 0.133789, a: 2.860633, b: 0.000000
Epoch 300, 평균 비용(Cost): 0.104196, a: 3.401512, b: 0.000000
Epoch 400, 평균 비용(Cost): 0.086192, a: 3.824389, b: 0.000000
Epoch 500, 평균 비용(Cost): 0.073786, a: 4.175776, b: 0.000000
Epoch 600, 평균 비용(Cost): 0.064613, a: 4.478100, b: 0.000000
Epoch 700, 평균 비용(Cost): 0.057512, a: 4.744184, b: 0.000000
Epoch 800, 평균 비용(Cost): 0.051834, a: 4.982181, b: 0.000000
Epoch 900, 평균 비용(Cost): 0.047181, a: 5.197650, b: 0.000000
Epoch 999, 평균 비용(Cost): 0.043330, a: 5.392711, b: 0.000000


In [12]:
# 학습 완료 후 최종 가중치와 편향 확인
print("학습된 a", a.item())
print("학습된 b", b.item())

학습된 a 5.3927106857299805
학습된 b 8.616920865733846e-08


In [ ]:
# 새로운 입력값 예측
# 키가 185cm인 사람이 농구선수인지 예측합니다.
input_height = 185

# 새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 합니다.
# 학습 때 사용한 X_mean, X_std 를 그대로 다시 사용합니다. (새로 계산하면 안 됩니다.)
input_norm = (input_height - X_mean) / X_std

# 예측은 학습이 아니므로 미분값을 계산할 필요가 없습니다.
# 그래서 torch.no_grad() 안에서 계산해, 불필요한 미분 기록을 만들지 않습니다.
with torch.no_grad():
    # 학습된 a, b는 tensor이므로, 입력값도 tensor로 맞춰서 계산합니다.
    input_norm_tensor = torch.tensor(input_norm, dtype=torch.float32)
    # H(x) = a * X_norm + b (선형 계산값  **확률이 아닙니다.)
    H_input = a * input_norm_tensor + b
    # z = sigmoid(H) (예측 확률 - 0~1 사이)
    probability = torch.sigmoid(H_input)

# 출력/판정할 때는 .item()으로 tensor에서 숫자만 꺼냅니다.
print(f"\n키가 {input_height}cm인 사람의 농구선수 확률: { probability.item():.4f}")

# 이진 분류에서는 보통 확률이 0.5 이상이면 1, 0.5 미만이면 0으로 판단합니다.
pred = 1 if probability.item() >= 0.5 else 0
if pred == 1:
    print("판별 결과: 농구선수입니다.")
else:
    print("판별 결과: 농구선수가 아닙니다.")